In [1]:
import sys
import os
import numpy as np
from functools import partial
from deap import base, creator, tools, algorithms

sys.path.append(os.path.abspath('..'))

os.makedirs("Resultados_Mochila_6obj", exist_ok=True)

from src.meamt_core import (
    build_toolbox_knapsack,
    create_knapsack_instance,
    evaluate_knapsack_individual,
    calcular_hv_unico_pymoo
)

NOBJ = 6 
INDSIZE = 100
NGEN = 600
P_DIVISIONS = 6
SEED = 42 

ref_points = tools.uniform_reference_points(nobj=NOBJ, p=P_DIVISIONS)
H = len(ref_points)
NPOP = H + (4 - H % 4) if H % 4 != 0 else H

VALORES, PESOS, CAPACIDADES, MAX_PROFITS = create_knapsack_instance(INDSIZE, NOBJ, seed=SEED)

func_eval_safe = partial(
    evaluate_knapsack_individual, 
    values=VALORES, 
    weights=PESOS, 
    capacities=CAPACIDADES, 
    max_profits=MAX_PROFITS
)

toolbox = build_toolbox_knapsack(func_eval_safe, INDSIZE, NPOP, NOBJ)



if __name__ == '__main__':
    for i in range(1, 31):
        pop = toolbox.population()
    
        fitnesses = toolbox.map(toolbox.evaluate, pop)
        for ind, fit in zip(pop, fitnesses):
            ind.fitness.values = fit

        logbook = tools.Logbook()
        logbook.header = "gen", "hypervolume"
        REF_POINT_HV = [0.0] * NOBJ

        for gen in range(1, NGEN + 1):
            offspring = algorithms.varAnd(pop, toolbox, cxpb=0.9, mutpb=1.0)
        
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
            for ind, fit in zip(invalid_ind, fitnesses):
                ind.fitness.values = fit

            pop.extend(offspring)
            pop = tools.selNSGA3(pop, NPOP, ref_points)

            hv_val = calcular_hv_unico_pymoo(pop, REF_POINT_HV)
            logbook.record(gen=gen, hypervolume=hv_val)
            
        fronteira_final = [ind.fitness.values for ind in pop]
        matriz_resultados = np.array(fronteira_final) 
        nome_arquivo = f"Resultados_Mochila_6obj/nsgaiii_run_{i}.npy"
        np.save(nome_arquivo, matriz_resultados)  
